# 13 - Custom Firmware Integration

**Objective:** Add custom Verilog/VHDL modules, integrate with AXI-lite interface, and rebuild the firmware. This notebook covers:
- Custom Verilog module development
- AXI-lite interface integration
- Firmware rebuild process
- Python interface to custom registers
- Debugging and verification

## Prerequisites
- Vivado installation (2020.1 or later)
- QICK source code
- Basic Verilog/VHDL knowledge
- Familiarity with AXI bus protocol

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging
import os
import subprocess

from qick import QickSoc

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

# Connect to the board to check current firmware info
BITSTREAM_PATH = '/path/to/current_firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)

# Get firmware information
cfg = soc.get_cfg()
# Get firmware information
cfg = soc.get_cfg()

print(f"Board model: {cfg.get('board', 'Unknown')}")
print(f"Software version: {cfg.get('sw_version', 'Unknown')}")
print(f"Firmware timestamp: {cfg.get('fw_timestamp', 'Unknown')}")

# In this version, the channels are in 'gens' and 'readouts'.
print(f"Available Generators (DACs): {len(cfg.get('gens', []))}")
print(f"Available Readouts (ADCs): {len(cfg.get('readouts', []))}")

# Useful Additional Information
if 'refclk_freq' in cfg:
    print(f"Reference Clock: {cfg['refclk_freq']} MHz")

# El tProc v2 suele estar en la lista de 'tprocs'
if 'tprocs' in cfg:
    print(f"Number of tProc cores: {len(cfg['tprocs'])}")

Board model: RFSoC4x2
Software version: 0.2.405
Firmware timestamp: Mon Apr  6 16:32:19 2026
Available Generators (DACs): 2
Available Readouts (ADCs): 3
Reference Clock: 491.52 MHz
Number of tProc cores: 1


## 2. Custom Verilog Module Example

Here's a complete example of a custom Verilog module with AXI-lite interface. This module demonstrates:
- AXI-lite slave interface for register access
- Configurable parameters
- Simple DSP operation (multiply-accumulate)
- Status registers for debugging

### 2.1 Custom Module Verilog Code

Create a file named `custom_dsp_module.v`:

In [ ]:
verilog_code = '''
// Custom DSP Module for QICK
// Supports multiply-accumulate operations with AXI-lite interface

module custom_dsp_module #(
    parameter AXI_ADDR_WIDTH = 32,
    parameter AXI_DATA_WIDTH = 32,
    parameter NUM_REGS = 8
)(
    input wire clk,
    input wire rst,
    
    // AXI-lite slave interface
    input wire [AXI_ADDR_WIDTH-1:0] s_axi_awaddr,
    input wire s_axi_awvalid,
    output wire s_axi_awready,
    input wire [AXI_DATA_WIDTH-1:0] s_axi_wdata,
    input wire s_axi_wvalid,
    output wire s_axi_wready,
    output wire [1:0] s_axi_bresp,
    output wire s_axi_bvalid,
    input wire s_axi_bready,
    input wire [AXI_ADDR_WIDTH-1:0] s_axi_araddr,
    input wire s_axi_arvalid,
    output wire s_axi_arready,
    output wire [AXI_DATA_WIDTH-1:0] s_axi_rdata,
    output wire [1:0] s_axi_rresp,
    output wire s_axi_rvalid,
    input wire s_axi_rready,
    
    // Custom outputs (can connect to DACs or other modules)
    output reg [31:0] custom_output,
    output reg [31:0] mac_result,
    
    // Status outputs
    output reg [31:0] status,
    output reg interrupt
);

    // Register addresses
    localparam REG_CONTROL   = 8'h00;
    localparam REG_STATUS    = 8'h04;
    localparam REG_VALUE_A   = 8'h08;
    localparam REG_VALUE_B   = 8'h0C;
    localparam REG_MAC_RESULT = 8'h10;
    localparam REG_CUSTOM_OUT = 8'h14;
    localparam REG_CONFIG    = 8'h18;
    localparam REG_VERSION   = 8'hFC;
    
    // Internal registers
    reg [AXI_DATA_WIDTH-1:0] regs [0:NUM_REGS-1];
    
    // Multiply-accumulate state
    reg [63:0] accumulator;
    reg [31:0] value_a, value_b;
    reg mac_busy;
    reg [7:0] mac_counter;
    
    // AXI state machine
    reg [1:0] axi_state;
    reg [AXI_ADDR_WIDTH-1:0] write_addr;
    
    // Initialize registers
    integer i;
    initial begin
        for (i = 0; i < NUM_REGS; i = i + 1) begin
            regs[i] = 32'd0;
        end
        regs[REG_VERSION >> 2] = 32'h00010001;  // Version 1.1
        accumulator = 64'd0;
        mac_busy = 1'b0;
        custom_output = 32'd0;
        mac_result = 32'd0;
        status = 32'd0;
        interrupt = 1'b0;
    end
    
    // AXI-lite write channel
    assign s_axi_awready = (axi_state == 2'b01);
    assign s_axi_wready = (axi_state == 2'b01);
    assign s_axi_bresp = 2'b00;
    assign s_axi_bvalid = (axi_state == 2'b10);
    
    // AXI-lite read channel
    assign s_axi_arready = (axi_state == 2'b00);
    assign s_axi_rdata = regs[s_axi_araddr[7:2]];
    assign s_axi_rresp = 2'b00;
    assign s_axi_rvalid = (axi_state == 2'b11);
    
    // AXI-lite state machine
    always @(posedge clk) begin
        if (rst) begin
            axi_state <= 2'b00;
            write_addr <= 32'd0;
        end else begin
            case (axi_state)
                2'b00: begin  // Idle
                    if (s_axi_awvalid && s_axi_wvalid) begin
                        write_addr <= s_axi_awaddr;
                        axi_state <= 2'b01;
                    end else if (s_axi_arvalid) begin
                        axi_state <= 2'b11;
                    end
                end
                2'b01: begin  // Write data received
                    regs[write_addr[7:2]] <= s_axi_wdata;
                    axi_state <= 2'b10;
                end
                2'b10: begin  // Write response
                    if (s_axi_bready) begin
                        axi_state <= 2'b00;
                    end
                end
                2'b11: begin  // Read response
                    if (s_axi_rready) begin
                        axi_state <= 2'b00;
                    end
                end
            endcase
        end
    end
    
    // Multiply-Accumulate operation
    always @(posedge clk) begin
        if (rst) begin
            accumulator <= 64'd0;
            mac_busy <= 1'b0;
            mac_counter <= 8'd0;
            mac_result <= 32'd0;
        end else begin
            // Check for MAC start command
            if (regs[REG_CONTROL >> 2][0] && !mac_busy) begin
                value_a <= regs[REG_VALUE_A >> 2];
                value_b <= regs[REG_VALUE_B >> 2];
                accumulator <= 64'd0;
                mac_counter <= regs[REG_CONFIG >> 2][7:0];
                mac_busy <= 1'b1;
                regs[REG_CONTROL >> 2][0] <= 1'b0;  // Clear start bit
                status[0] <= 1'b1;  // MAC busy
            end
            
            // Perform MAC iterations
            if (mac_busy && (mac_counter > 0)) begin
                accumulator <= accumulator + (value_a * value_b);
                mac_counter <= mac_counter - 1;
            end
            
            // MAC complete
            if (mac_busy && (mac_counter == 0)) begin
                mac_busy <= 1'b0;
                mac_result <= accumulator[31:0];
                regs[REG_MAC_RESULT >> 2] <= accumulator[31:0];
                status[0] <= 1'b0;  // MAC done
                status[1] <= 1'b1;  // MAC complete flag
                interrupt <= 1'b1;
            end
            
            // Clear interrupt on read
            if (s_axi_arvalid && (s_axi_araddr[7:2] == (REG_STATUS >> 2))) begin
                interrupt <= 1'b0;
                status[1] <= 1'b0;
            end
        end
    end
    
    // Custom output from register
    always @(posedge clk) begin
        if (rst) begin
            custom_output <= 32'd0;
        end else begin
            custom_output <= regs[REG_CUSTOM_OUT >> 2];
        end
    end
    
endmodule
'''

print("=== Custom Verilog Module ===")
print(verilog_code[:500] + "...")
print(f"\nTotal module size: {len(verilog_code)} characters")
print("\nKey features:")
print("  - AXI-lite slave interface for register access")
print("  - Multiply-accumulate (MAC) operation")
print("  - Status and interrupt generation")
print("  - Configurable parameters")

### 2.2 Integration Steps

Follow these steps to integrate the custom module into the QICK firmware.

In [ ]:
def print_integration_steps():
    """Print the firmware integration procedure."""
    
    steps = """
    ========================================
    STEP 1: Add Module to QICK Source
    ========================================
    
    # Copy the Verilog file to the QICK IP directory
    cp custom_dsp_module.v $QICK_PATH/firmware/ip/
    
    # Add the module to the IP catalog (create a simple wrapper if needed)
    cd $QICK_PATH/firmware
    
    ========================================
    STEP 2: Update Block Design (TCL Script)
    ========================================
    
    # Create a TCL script to add the module to the block design
    cat > add_custom_module.tcl << 'EOF'
    # Create and configure custom module cell
    create_bd_cell -type ip -vlnv user:user:custom_dsp_module:1.0 custom_dsp_module_0
    
    # Connect AXI-lite interface to interconnect
    connect_bd_intf_net -intf_net axi_interconnect_0_m00_axi \\
        [get_bd_intf_pins custom_dsp_module_0/s_axi]
    
    # Connect clocks and resets
    connect_bd_net -net clk_wiz_0_clk_out1 \\
        [get_bd_pins custom_dsp_module_0/clk]
    connect_bd_net -net rst_clk_wiz_0_100M_peripheral_aresetn \\
        [get_bd_pins custom_dsp_module_0/rst]
    
    # Assign address range (check available address space)
    assign_bd_address -target_address_space /processing_system7_0/Data \\
        [get_bd_addr_segs custom_dsp_module_0/s_axi/reg0] \\
        -range 0x1000 -offset 0x43C00000
    EOF
    
    # Run TCL script in Vivado
    vivado -source add_custom_module.tcl $QICK_PATH/firmware/projects/qick_tprocv2_4x2_standard_b_5/qick_tprocv2_4x2_standard_b_5.xpr
    
    ========================================
    STEP 3: Rebuild Firmware
    ========================================
    
    cd $QICK_PATH/firmware
    make clean
    make all
    
    # The new bitstream will be in:
    # projects/qick_tprocv2_4x2_standard_b_5/out/qick_4x2.bit
    """
    
    print(steps)

print_integration_steps()

## 3. Python Interface to Custom Module

Once the firmware is rebuilt with your custom module, you can interface with it from Python using AXI-lite reads and writes.

In [ ]:
class CustomDSPModule:
    """Python interface for the custom DSP module."""
    
    # Register offsets (must match Verilog defines)
    REG_CONTROL = 0x00
    REG_STATUS = 0x04
    REG_VALUE_A = 0x08
    REG_VALUE_B = 0x0C
    REG_MAC_RESULT = 0x10
    REG_CUSTOM_OUT = 0x14
    REG_CONFIG = 0x18
    REG_VERSION = 0xFC
    
    def __init__(self, soc, base_addr=0x43C00000):
        """
        Initialize the custom module interface.
        
        Parameters:
        - soc: QickSoc instance
        - base_addr: AXI base address of the custom module
        """
        self.soc = soc
        self.base = base_addr
        self._check_version()
    
    def _check_version(self):
        """Verify the module is present and get version."""
        try:
            version = self.read_reg(self.REG_VERSION)
            major = (version >> 16) & 0xFFFF
            minor = version & 0xFFFF
            print(f"Custom module detected: version {major}.{minor}")
        except Exception as e:
            print(f"Warning: Could not read custom module version: {e}")
    
    def write_reg(self, offset, value):
        """Write to a register."""
        self.soc.axilite_write(self.base + offset, value)
    
    def read_reg(self, offset):
        """Read from a register."""
        return self.soc.axilite_read(self.base + offset)
    
    def set_value_a(self, value):
        """Set value A for MAC operation."""
        self.write_reg(self.REG_VALUE_A, value & 0xFFFFFFFF)
    
    def set_value_b(self, value):
        """Set value B for MAC operation."""
        self.write_reg(self.REG_VALUE_B, value & 0xFFFFFFFF)
    
    def set_custom_output(self, value):
        """Set the custom output value."""
        self.write_reg(self.REG_CUSTOM_OUT, value & 0xFFFFFFFF)
    
    def set_config(self, num_iterations):
        """Configure MAC operation parameters."""
        self.write_reg(self.REG_CONFIG, num_iterations & 0xFF)
    
    def start_mac(self):
        """Start the multiply-accumulate operation."""
        self.write_reg(self.REG_CONTROL, 0x01)
    
    def is_mac_busy(self):
        """Check if MAC operation is in progress."""
        status = self.read_reg(self.REG_STATUS)
        return (status & 0x01) != 0
    
    def get_mac_result(self):
        """Get the MAC result (blocking until complete)."""
        # Wait for completion (or check interrupt)
        import time
        timeout = 0.001  # 1 ms timeout
        start_time = time.time()
        
        while self.is_mac_busy():
            if time.time() - start_time > timeout:
                raise TimeoutError("MAC operation timed out")
        
        return self.read_reg(self.REG_MAC_RESULT)
    
    def multiply_accumulate(self, a, b, num_iterations=1):
        """
        Perform a multiply-accumulate operation on hardware.
        
        Returns a * b * num_iterations (in hardware, with 32-bit precision)
        """
        self.set_value_a(a)
        self.set_value_b(b)
        self.set_config(num_iterations)
        self.start_mac()
        return self.get_mac_result()
    
    def get_status_string(self):
        """Get human-readable status."""
        status = self.read_reg(self.REG_STATUS)
        bits = []
        if status & 0x01:
            bits.append("MAC_BUSY")
        if status & 0x02:
            bits.append("MAC_DONE")
        return f"Status: {', '.join(bits) if bits else 'IDLE'} (0x{status:08X})"


# Example usage (after firmware is rebuilt)
def example_custom_module_usage():
    """Example of using the custom module."""
    
    print("=== Custom Module Usage Example ===\n")
    
    # Initialize the custom module (use correct base address)
    # custom = CustomDSPModule(soc, base_addr=0x43C00000)
    
    # Test basic operations
    # custom.set_custom_output(0xDEADBEEF)
    # print(f"Custom output set to: 0x{custom.read_reg(CustomDSPModule.REG_CUSTOM_OUT):08X}")
    
    # Perform MAC operation
    # result = custom.multiply_accumulate(1000, 2000, num_iterations=5)
    # print(f"MAC result: {result}")
    # Expected: 1000 * 2000 * 5 = 10,000,000
    
    print("Example operations:")
    print("  - Set custom output: custom.set_custom_output(0xDEADBEEF)")
    print("  - Read status: custom.get_status_string()")
    print("  - MAC operation: result = custom.multiply_accumulate(1000, 2000, 5)")

example_custom_module_usage()

## 4. Debugging Custom Firmware

Here are techniques for debugging custom firmware modules.

In [ ]:
def debug_firmware_techniques():
    """Debugging techniques for custom firmware."""
    
    print("=== Firmware Debugging Techniques ===\n")
    
    print("1. Integrated Logic Analyzer (ILA)")
    print("   - Add ILA cores to capture internal signals")
    print("   - Use Vivado hardware manager to view waveforms")
    print("   - Example TCL:")
    print("     create_bd_cell -type ip -vlnv xilinx.com:ip:ila:6.2 ila_0")
    print("     connect_bd_net [get_bd_pins custom_module/signal] [get_bd_pins ila_0/probe0]\n")
    
    print("2. Virtual I/O (VIO) for Real-Time Control")
    print("   - Use VIO to read/write signals during runtime")
    print("   - Example TCL:")
    print("     create_bd_cell -type ip -vlnv xilinx.com:ip:vio:3.0 vio_0\n")
    
    print("3. AXI Verification")
    print("   - Use AXI VIP for protocol checking")
    print("   - Simulate with XSIM or ModelSim")
    print("   - Check address mapping in the address editor\n")
    
    print("4. Python-Based Debugging")
    print("   - Read/write registers to verify functionality")
    print("   - Use timeout-based polling for status bits")
    print("   - Implement diagnostic functions")
    print("")
    
    print("5. Common Issues and Solutions")
    issues = [
        ("Clock domain crossing", "Use proper synchronizers or CDC modules"),
        ("AXI address conflict", "Check address ranges in address editor"),
        ("Timing closure failures", "Add pipeline registers or reduce logic depth"),
        ("Missing reset connections", "Verify all modules receive reset signals"),
        ("Incorrect AXI handshake", "Follow AXI-lite protocol specification")
    ]
    
    for issue, solution in issues:
        print(f"   - {issue}: {solution}")

debug_firmware_techniques()

## 5. Firmware Verification Test

After rebuilding the firmware, run this verification test to ensure everything works.

In [ ]:
def verify_custom_firmware(soc, custom_module, expected_version="1.0"):
    """
    Comprehensive verification test for custom firmware.
    
    Parameters:
    - soc: QickSoc instance
    - custom_module: CustomDSPModule instance
    - expected_version: Expected version string
    
    Returns:
    - True if all tests pass, False otherwise
    """
    print("=== Firmware Verification ===\n")
    
    tests_passed = 0
    tests_total = 0
    
    # Test 1: Version check
    tests_total += 1
    try:
        version = custom_module.read_reg(CustomDSPModule.REG_VERSION)
        version_str = f"{(version >> 16) & 0xFFFF}.{version & 0xFFFF}"
        print(f"✓ Version: {version_str}")
        tests_passed += 1
    except Exception as e:
        print(f"✗ Version check failed: {e}")
    
    # Test 2: Register read/write
    tests_total += 1
    test_value = 0x12345678
    try:
        custom_module.write_reg(CustomDSPModule.REG_CUSTOM_OUT, test_value)
        read_value = custom_module.read_reg(CustomDSPModule.REG_CUSTOM_OUT)
        if read_value == test_value:
            print(f"✓ Register write/read: OK (0x{read_value:08X})")
            tests_passed += 1
        else:
            print(f"✗ Register mismatch: wrote 0x{test_value:08X}, read 0x{read_value:08X}")
    except Exception as e:
        print(f"✗ Register test failed: {e}")
    
    # Test 3: MAC operation
    tests_total += 1
    try:
        a, b, n = 100, 200, 3
        expected = a * b * n
        result = custom_module.multiply_accumulate(a, b, n)
        if result == expected:
            print(f"✓ MAC test: {a} * {b} * {n} = {result}")
            tests_passed += 1
        else:
            print(f"✗ MAC test: expected {expected}, got {result}")
    except Exception as e:
        print(f"✗ MAC test failed: {e}")
    
    # Test 4: Basic QICK functionality
    tests_total += 1
    try:
        cfg = soc.get_cfg()
        print(f"✓ QICK readout: {cfg['adcs']} ADCs, {cfg['dacs']} DACs")
        tests_passed += 1
    except Exception as e:
        print(f"✗ QICK test failed: {e}")
    
    print(f"\nTest Results: {tests_passed}/{tests_total} passed")
    return tests_passed == tests_total

print("Verification function ready.")
print("After rebuilding firmware, run:")
print("  custom = CustomDSPModule(soc, base_addr=0x43C00000)")
print("  verify_custom_firmware(soc, custom)")

## 6. Resources and References

In [ ]:
def print_resources():
    """Print useful resources for custom firmware development."""
    
    resources = {
        "QICK Documentation": "https://qick.readthedocs.io/en/latest/firmware.html",
        "Vivado AXI Reference": "https://www.xilinx.com/support/documentation/ip_documentation/axi_ref_guide/latest/ug1037-vivado-axi-reference-guide.pdf",
        "QICK GitHub": "https://github.com/openquantumhardware/qick",
        "Xilinx Vivado Documentation": "https://www.xilinx.com/support/documentation-navigation/design-hubs/dh004-ultrascale-plus-vivado-design-hub.html",
        "AXI-lite Protocol": "https://developer.arm.com/documentation/ihi0022/latest/"
    }
    
    print("=== Useful Resources ===\n")
    for name, url in resources.items():
        print(f"- {name}: {url}")
    
    print("\n=== Key Commands Reference ===\n")
    cmds = [
        ("make all", "Build firmware"),
        ("make clean", "Clean build artifacts"),
        ("vivado -source script.tcl project.xpr", "Run TCL script in Vivado"),
        ("report_ip_status", "Check IP status in Vivado"),
        ("validate_bd_design", "Validate block design"),
        ("write_bitstream", "Generate bitstream"),
    ]
    
    for cmd, desc in cmds:
        print(f"  {cmd}: {desc}")

print_resources()

## 7. Summary

You have learned:
- How to create custom Verilog modules with AXI-lite interfaces
- Integration steps for adding custom modules to QICK firmware
- How to rebuild the firmware with custom modules
- Python interface development for custom registers
- Debugging techniques for custom firmware
- Verification procedures for new firmware

**Key takeaways:**
- Custom modules extend QICK functionality for specialized applications
- AXI-lite provides a standard interface for register access
- Use Vivado's IP integrator for block design modifications
- Always verify custom firmware with test routines
- Use ILA/VIO cores for hardware debugging

**Next steps:**

- For multi-board synchronization without XCOM hardware, see [`10_Multi_Board_Synchronization.ipynb`](./10_Multi_Board_Synchronization.ipynb)
- For advanced multi-board networking with XCOM, see [`14_XCOM_Network_Synchronization.ipynb`](./14_XCOM_Network_Synchronization.ipynb)
- Review the QICK firmware documentation for detailed build instructions
- Experiment with simple custom modules before complex designs
- Join the QICK community for support and collaboration